<a href="https://colab.research.google.com/github/polinas2307/labs/blob/main/lab4_spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота №4. Розподілена обробка даних та паралелізм (Apache Spark)
**Курс:** «Основи інженерії даних», 2025/26 н.р.  
**Задача:** обчислити середню тривалість сеансу на великому лог-наборі, порівняти стратегії розподілу даних (вплив shuffle), оцінити прискорення за законом Амдала, проаналізувати перекіс (data skew) за `user_id`.

**Дані:** синтетичний лог-набір Parquet зі схемою із завдання: `user_id: STRING`, `session_id: STRING`, `event_timestamp: LONG` (мс). ≈3.75 млн подій, 250 тис. сеансів, 60 тис. користувачів (генератор: `gen_data.py`, зерно `42`). Окремо згенеровано набір зі скошеністю, де 90% сеансів належать одному користувачу.

**Середовище:** PySpark (локальний режим `local[N]`, один виконавець, N ядер). Адаптивне виконання запитів (AQE) вимкнено, щоб результати порівняння були чистими.

In [5]:
!pip install -q pyspark pyarrow pandas matplotlib
!python gen_data.py

normal rows 3747859
skew rows 3747027


## 1. Налаштування Spark та допоміжні функції

In [6]:
import os, time, json, urllib.request, statistics
from pyspark.sql import SparkSession, functions as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA, DATA_SKEW = "data/sessions", "data/sessions_skew"
RESULTS = {}

def make_spark(cores):
    # Типові значення: X = 1 виконавець (local-режим), Y = cores ядер на виконавця
    return (SparkSession.builder.master(f"local[{cores}]").appName(f"lab4-{cores}")
            .config("spark.driver.memory", "2g")
            .config("spark.sql.adaptive.enabled", "false")      # AQE вимкнено для чистого експерименту
            .config("spark.sql.shuffle.partitions", "200")      # типове значення Spark
            .config("spark.ui.showConsoleProgress", "false")
            .config("spark.ui.port", "4040").getOrCreate())

def stage_totals(spark):
    """Сумарні метрики всіх стадій через REST API Spark UI: байти запису shuffle та executorRunTime."""
    sc = spark.sparkContext
    url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/stages"
    st = json.load(urllib.request.urlopen(url))
    return (sum(s["shuffleWriteBytes"] for s in st), sum(s["executorRunTime"] for s in st))

def measure(spark, fn, repeats=3):
    fn()                                   # прогрів (JIT, ініціалізація)
    rows = []
    for _ in range(repeats):
        b0, r0 = stage_totals(spark)
        t0 = time.perf_counter(); res = fn(); t = time.perf_counter() - t0
        time.sleep(1.5)                    # дати UI зібрати метрики
        b1, r1 = stage_totals(spark)
        rows.append(dict(time=t, shuffle_mb=(b1-b0)/1e6, run_s=(r1-r0)/1e3, result=res))
    med = lambda k: statistics.median(r[k] for r in rows)
    return dict(time=med("time"), time_all=[r["time"] for r in rows],
                shuffle_mb=med("shuffle_mb"), run_s=med("run_s"), result=rows[0]["result"])

## 2. Алгоритми

Спільні кроки: **читання → groupBy(`user_id`, `session_id`) → `MAX(ts) − MIN(ts)` → загальне середнє**. Читання з диска винесено за межі виміру: вхідні дані кешуються і матеріалізуються до початку таймера.

| Стратегія | Опис | Кількість shuffle сирих даних |
|---|---|---|
| **N. Наївна (типові налаштування)** | `groupBy(...)` при `shuffle.partitions=200` | 1 (але з map-side combine: летить уже частково агрегований вихід) |
| **A. Високе перетасування** | `repartition(200)` (round-robin) перед `groupBy` | 2 (повне переміщення сирих подій + другий shuffle) |
| **B. Низьке перетасування** | `repartition(P, user_id, session_id)`, де P = кількість ядер; `shuffle.partitions = P`, `groupBy` по тих самих ключах | 1 (другий Exchange усувається оптимізатором) |
| **C. Попередньо розподілені дані** | дані вже розбиті за ключами на P розділів (підготовка поза виміром), агрегація без shuffle | 0 в основній агрегації |

In [7]:
def avg_duration(df):
    return (df.groupBy("user_id", "session_id")
              .agg((F.max("event_timestamp") - F.min("event_timestamp")).alias("dur_ms"))
              .agg(F.avg("dur_ms").alias("avg_ms")).first()["avg_ms"])

def strat_N(spark, df, P):
    spark.conf.set("spark.sql.shuffle.partitions", "200")
    return lambda: avg_duration(df)

def strat_A(spark, df, P):
    spark.conf.set("spark.sql.shuffle.partitions", "200")
    return lambda: avg_duration(df.repartition(200))

def strat_B(spark, df, P):
    spark.conf.set("spark.sql.shuffle.partitions", str(P))
    return lambda: avg_duration(df.repartition(P, "user_id", "session_id"))

def strat_C(spark, df, P):
    spark.conf.set("spark.sql.shuffle.partitions", str(P))
    pre = df.repartition(P, "user_id", "session_id").cache(); pre.count()   # підготовка (одноразово)
    return lambda: avg_duration(pre)

STRATS = {"N: наївна (default)": strat_N, "A: високий shuffle": strat_A,
          "B: низький shuffle": strat_B, "C: pre-partitioned": strat_C}

### Перевірка планів виконання (чи справді B усуває другий Exchange)

In [8]:
spark = make_spark(2)
df = spark.read.parquet(DATA).cache(); n = df.count()
print("Рядків:", n, "| розділів при читанні:", df.rdd.getNumPartitions())
spark.conf.set("spark.sql.shuffle.partitions", "2")
agg = lambda d: d.groupBy("user_id","session_id").agg((F.max("event_timestamp")-F.min("event_timestamp")).alias("d"))
print("=== A (round-robin repartition) ==="); agg(df.repartition(200)).explain()
print("=== B (repartition за ключами) ===");   agg(df.repartition(2,"user_id","session_id")).explain()

Рядків: 3747859 | розділів при читанні: 3
=== A (round-robin repartition) ===
== Physical Plan ==
*(2) HashAggregate(keys=[user_id#0, session_id#1], functions=[max(event_timestamp#2L), min(event_timestamp#2L)])
+- Exchange hashpartitioning(user_id#0, session_id#1, 2), ENSURE_REQUIREMENTS, [plan_id=54]
   +- *(1) HashAggregate(keys=[user_id#0, session_id#1], functions=[partial_max(event_timestamp#2L), partial_min(event_timestamp#2L)])
      +- Exchange RoundRobinPartitioning(200), REPARTITION_BY_NUM, [plan_id=46]
         +- InMemoryTableScan [user_id#0, session_id#1, event_timestamp#2L]
               +- InMemoryRelation [user_id#0, session_id#1, event_timestamp#2L], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- *(1) ColumnarToRow
                        +- FileScan parquet [user_id#0,session_id#1,event_timestamp#2L] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/data/sessions], PartitionFilters: [], Pu

## 3. Експеримент: 4 стратегії × 3 конфігурації ядер (1, 2, 4), медіана з 3 запусків

In [9]:
results = {}
for cores in [1, 2, 4]:
    if "spark" in globals():
        try: spark.stop()
        except Exception: pass
    spark = make_spark(cores)
    df = spark.read.parquet(DATA).cache(); df.count()          # читання поза виміром
    for name, mk in STRATS.items():
        fn = mk(spark, df, cores)
        results[(cores, name)] = measure(spark, fn)
        r = results[(cores, name)]
        print(f"cores={cores} | {name:24s} | {r['time']:6.2f} с | shuffle {r['shuffle_mb']:7.1f} МБ | avg={r['result']/60000:.3f} хв")
    df.unpersist()

cores=1 | N: наївна (default)      |   6.14 с | shuffle     7.1 МБ | avg=30.495 хв
cores=1 | A: високий shuffle       |  24.41 с | shuffle   203.4 МБ | avg=30.495 хв
cores=1 | B: низький shuffle       |   3.29 с | shuffle    29.3 МБ | avg=30.495 хв
cores=1 | C: pre-partitioned       |   1.88 с | shuffle     0.0 МБ | avg=30.495 хв
cores=2 | N: наївна (default)      |   3.61 с | shuffle     7.1 МБ | avg=30.495 хв
cores=2 | A: високий shuffle       |  19.75 с | shuffle   186.4 МБ | avg=30.495 хв
cores=2 | B: низький shuffle       |   3.98 с | shuffle    29.4 МБ | avg=30.495 хв
cores=2 | C: pre-partitioned       |   1.76 с | shuffle     0.0 МБ | avg=30.495 хв
cores=4 | N: наївна (default)      |   3.51 с | shuffle     7.1 МБ | avg=30.495 хв
cores=4 | A: високий shuffle       |  19.37 с | shuffle   186.4 МБ | avg=30.495 хв
cores=4 | B: низький shuffle       |   3.28 с | shuffle    29.4 МБ | avg=30.495 хв
cores=4 | C: pre-partitioned       |   1.36 с | shuffle     0.0 МБ | avg=30.495 хв


In [10]:
import pandas as pd
tab = pd.DataFrame([dict(cores=c, strategy=s, time_s=round(r["time"],2), shuffle_MB=round(r["shuffle_mb"],1),
                         task_time_s=round(r["run_s"],2), avg_session_min=round(r["result"]/60000,4))
                    for (c,s),r in results.items()])
RESULTS["table"] = tab.to_dict("records"); tab

,cores,strategy,time_s,shuffle_MB,task_time_s,avg_session_min
0,1,N: наївна (default),6.14,7.1,3.90,30.4949
1,1,A: високий shuffle,24.41,203.4,21.70,30.4949
2,1,B: низький shuffle,3.29,29.3,3.10,30.4949
3,1,C: pre-partitioned,1.88,0.0,1.77,30.4949
4,2,N: наївна (default),3.61,7.1,4.34,30.4949
5,2,A: високий shuffle,19.75,186.4,35.14,30.4949
6,2,B: низький shuffle,3.98,29.4,7.03,30.4949
7,2,C: pre-partitioned,1.76,0.0,3.23,30.4949
8,4,N: наївна (default),3.51,7.1,7.63,30.4949
9,4,A: високий shuffle,19.37,186.4,64.37,30.4949


In [11]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
piv_t = tab.pivot(index="strategy", columns="cores", values="time_s"); piv_s = tab.pivot(index="strategy", columns="cores", values="shuffle_MB")
piv_t.plot.bar(ax=ax[0], rot=20); ax[0].set_title("Час виконання агрегації, с (без читання)"); ax[0].set_ylabel("с")
piv_s.plot.bar(ax=ax[1], rot=20); ax[1].set_title("Shuffle write, МБ"); ax[1].set_ylabel("МБ")
plt.tight_layout(); plt.savefig("fig_strategies.png", dpi=130); plt.show()

## 4. Моделювання продуктивності: закон Амдала

$$S(p)=\frac{1}{s+\frac{1-s}{p}},\qquad S_{max}=\frac{1}{s}$$

де *s* — послідовна частка роботи. Оцінюємо її двома способами:

1. **З профілю одноядерного запуску (`local[1]`)**: паралельна робота = сумарний `executorRunTime` усіх задач, послідовна = решта часу (планування запиту, DAG-планувальник, збирання результату на драйвері): $s_{prof}=1-\frac{\sum executorRunTime}{T_1}$.
2. **З виміряного прискорення**: $s_{meas}=\dfrac{1/S(p)-1/p}{1-1/p}$, де $S(p)=T_1/T_p$.



In [12]:
import numpy as np
amd = {}
for name in STRATS:
    T1 = results[(1,name)]["time"]; R1 = results[(1,name)]["run_s"]
    s_prof = float(np.clip(1 - R1/T1, 0.0, 1.0))
    meas = {}
    for p in [2,4]:
        S = T1/results[(p,name)]["time"]
        meas[p] = dict(S=S, s=(1/S - 1/p)/(1 - 1/p))
    amd[name] = dict(T1=T1, R1=R1, s_prof=s_prof, Smax=(1/s_prof if s_prof>0 else float("inf")), meas=meas)
    print(f"{name:24s} T1={T1:5.2f}с  паралельна робота={R1:5.2f}с  s_prof={s_prof:.3f}  S_max={amd[name]['Smax']:.1f}x  | виміряно S(2)={meas[2]['S']:.2f}, S(4)={meas[4]['S']:.2f}")
RESULTS["amdahl"] = {k:{kk:vv for kk,vv in v.items()} for k,v in amd.items()}

N: наївна (default)      T1= 6.14с  паралельна робота= 3.90с  s_prof=0.365  S_max=2.7x  | виміряно S(2)=1.70, S(4)=1.75
A: високий shuffle       T1=24.41с  паралельна робота=21.70с  s_prof=0.111  S_max=9.0x  | виміряно S(2)=1.24, S(4)=1.26
B: низький shuffle       T1= 3.29с  паралельна робота= 3.10с  s_prof=0.058  S_max=17.3x  | виміряно S(2)=0.83, S(4)=1.00
C: pre-partitioned       T1= 1.88с  паралельна робота= 1.77с  s_prof=0.057  S_max=17.4x  | виміряно S(2)=1.07, S(4)=1.39


In [13]:
ps = np.array([1,2,4,8,16,32,64,128])
rows = {name: [1/(a["s_prof"] + (1-a["s_prof"])/p) for p in ps] for name,a in amd.items()}
amd_tab = pd.DataFrame(rows, index=[f"p={p}" for p in ps]).round(2); display(amd_tab)
RESULTS["amdahl_table"] = amd_tab.to_dict()
plt.figure(figsize=(7,4))
for name,v in rows.items(): plt.plot(ps, v, marker="o", label=name)
plt.plot(ps, ps, "k--", label="ідеальне"); plt.xscale("log", base=2); plt.yscale("log", base=2)
plt.xlabel("p (ядер)"); plt.ylabel("S(p)"); plt.title("Закон Амдала за s_prof"); plt.legend(fontsize=7); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig_amdahl.png", dpi=130); plt.show()

,N: наївна (default),A: високий shuffle,B: низький shuffle,C: pre-partitioned
p=1,1.00,1.00,1.00,1.00
p=2,1.47,1.80,1.89,1.89
p=4,1.91,3.00,3.41,3.41
p=8,2.25,4.50,5.69,5.71
p=16,2.47,6.00,8.56,8.59
p=32,2.60,7.20,11.45,11.51
p=64,2.67,8.00,13.77,13.85
p=128,2.71,8.47,15.32,15.43


## 5. Асиметрія даних (data skew): 1 користувач = 90% усіх сеансів

Окремий набір `sessions_skew`. Ключ `(user_id, session_id)` рівномірний, тому основна задача не страждає. Але будь-яка агрегація **по `user_id`** (середня тривалість сеансу на користувача, join з таблицею користувачів) відправить 90% рядків в один розділ. Нижче порівнюємо наївний підхід і **salting** (двоетапна агрегація з випадковою «сіллю»).

In [14]:
spark.stop(); spark = make_spark(4)
spark.conf.set("spark.sql.shuffle.partitions", "8")
ev = spark.read.parquet(DATA_SKEW)
sess = (ev.groupBy("user_id","session_id").agg((F.max("event_timestamp")-F.min("event_timestamp")).alias("dur_ms"))).cache()
print("сеансів:", sess.count())
top = sess.groupBy("user_id").count().orderBy(F.desc("count")).limit(3).toPandas(); print(top)
share = top["count"].iloc[0] / sess.count(); print(f"частка сеансів найбільшого користувача: {share:.1%}")

сеансів: 250000
  user_id   count
0      u0  224919
1  u38651       6
2   u6872       5
частка сеансів найбільшого користувача: 90.0%


In [15]:
SALT = 16
def part_sizes(d): return sorted([r["n"] for r in d.groupBy(F.spark_partition_id().alias("p")).agg(F.count("*").alias("n")).collect()], reverse=True)

naive_df  = sess.repartition(8, "user_id")
salted_df = sess.withColumn("salt", (F.rand(7)*SALT).cast("int")).repartition(8, "user_id", "salt")
sz_naive, sz_salt = part_sizes(naive_df), part_sizes(salted_df)
print("розміри розділів (наївно):", sz_naive); print("розміри розділів (salting):", sz_salt)

def per_user_naive():
    return sess.groupBy("user_id").agg(F.avg("dur_ms").alias("a")).agg(F.avg("a")).first()[0]
def per_user_salted():
    s = sess.withColumn("salt", (F.rand(7)*SALT).cast("int"))
    p1 = s.groupBy("user_id","salt").agg(F.sum("dur_ms").alias("s"), F.count("*").alias("c"))          # етап 1: часткова
    p2 = p1.groupBy("user_id").agg((F.sum("s")/F.sum("c")).alias("a"))                                  # етап 2: підсумок
    return p2.agg(F.avg("a")).first()[0]

sk = {"naive": measure(spark, per_user_naive), "salted": measure(spark, per_user_salted)}
for k,v in sk.items(): print(f"{k:7s} {v['time']:.2f} с | shuffle {v['shuffle_mb']:.2f} МБ | результат {v['result']/60000:.4f} хв")
assert abs(sk["naive"]["result"]-sk["salted"]["result"]) < 1e-3*sk["naive"]["result"], "результати мають збігатись"
RESULTS["skew"] = dict(share=float(share), part_naive=sz_naive, part_salt=sz_salt,
                       naive_time=sk["naive"]["time"], salted_time=sk["salted"]["time"])

розміри розділів (наївно): [228103, 3206, 3188, 3179, 3170, 3092, 3087, 2975]
розміри розділів (salting): [45593, 45398, 45141, 31441, 31217, 31001, 17048, 3161]
naive   0.34 с | shuffle 0.34 МБ | результат 30.3390 хв
salted  0.48 с | shuffle 0.83 МБ | результат 30.3390 хв


In [16]:
fig, ax = plt.subplots(1,2, figsize=(11,3.6))
ax[0].bar(range(8), sz_naive); ax[0].set_title("Розміри розділів: repartition(user_id)"); ax[0].set_xlabel("розділ"); ax[0].set_ylabel("рядків")
ax[1].bar(range(8), sz_salt, color="tab:green"); ax[1].set_title("Розміри розділів: user_id + salt"); ax[1].set_xlabel("розділ")
ax[1].set_ylim(0, max(sz_naive)*1.05); plt.tight_layout(); plt.savefig("fig_skew.png", dpi=130); plt.show()
json.dump(RESULTS, open("results.json","w"), default=float, ensure_ascii=False, indent=1)

## 6. Висновки та пом'якшення перекосу

**Як виявити перекіс (user_id = 90% сеансів):**
- `groupBy("user_id").count()` / `approx_count_distinct` + топ-N частот: один ключ різко домінує;
- Spark UI → вкладка Stages: у стадії shuffle одна задача працює на порядки довше за медіану, а в метриках *Shuffle Read Size* вона найбільша; Summary Metrics показує великий розрив max/median;
- `spark_partition_id()` + підрахунок рядків на розділ (як вище).

**Як пом'якшити:**
1. **Salting** — додати випадкову «сіль» до ключа та агрегувати у два етапи (реалізовано вище, результат ідентичний);
2. **AQE** (`spark.sql.adaptive.enabled=true`, `skewJoin.enabled=true`) — автоматично розбиває великі розділи під час join; для `groupBy` допомагає тільки з об'єднанням малих розділів;
3. **Broadcast join**, якщо друга таблиця мала — shuffle великої сторони взагалі зникає;
4. **Виокремлення «гарячого» ключа**: обробити користувача з 90% окремо (власна агрегація/кластер), решту — стандартно, потім `union`;
5. **Попередня агрегація на стороні джерела** (map-side combine, `reduceByKey` замість `groupByKey`) — зменшує обсяг даних, що йде в shuffle.